# 00 — Data Preparation

Loads `data/metadata/main_metadata.csv`, parses dates, standardizes water types, joins measurement row counts, and saves `data/analysis/processed_metadata.parquet` for use by all downstream inventory notebooks.

Run this notebook first before any other notebook in `code/03_inventory/`.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

root = Path('../..')
metadata_path  = root / 'data/metadata/main_metadata.csv'
counts_path    = root / 'data/analysis/csv_row_counts.csv'
output_path    = root / 'data/analysis/processed_metadata.parquet'

df = pd.read_csv(metadata_path, low_memory=False)
df['meta_row'] = np.arange(len(df))   # row position in main_metadata.csv (key for locations.csv)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

Loaded 109,440 rows, 74 columns


## Dates (table-level, 2026-11-04)

Each metadata entry takes the dates of its own table(s): `data/analysis/entry_dates.csv` from `15_entry_dates.py`, built on `table_dates.csv` (`14_table_dates.py`).
`date_status` names the evidence: table_date_column, table_text, table_caption, page_text, or a labeled last-resort fallback
(period_of_record, construction_year, llm_grounded), else undated. Valid years are 1800 to the publication year.
The LLM-string dates (protocol of 2026-10-28) are kept as `year_start_llm`, `year_end_llm`, `date_status_llm`.
`is_dated(df)` counts fallbacks as dated; `is_dated(df, include_fallbacks=False)` is the strict version.

In [2]:
import sys
sys.path.insert(0, '.')
from date_parsing import apply_entry_dates, is_dated, FALLBACK_STATUSES

# Table-level dates (2026-11-04): each entry takes the dates of its own table(s)
# (code/02_inventory/15_entry_dates.py). LLM-string dates kept as *_llm columns.
df = apply_entry_dates(df, root / 'data/analysis/entry_dates.csv')

print(df['date_status'].value_counts().to_string())
dated = is_dated(df)
print(f'Dated: {dated.sum():,} of {len(df):,} rows ({dated.mean():.1%}); undated {(~dated).sum():,}')
print(f"Fallback tiers (period of record / construction / grounded LLM): {df['date_status'].isin(FALLBACK_STATUSES).sum():,}")


date_status
table_caption        41667
table_date_column    31441
undated              15919
table_text           14369
llm_grounded          1834
construction_year     1828
page_text             1428
period_of_record       954
Dated: 93,521 of 109,440 rows (85.5%); undated 15,919
Fallback tiers (period of record / construction / grounded LLM): 4,616


## Standardize water types

In [3]:
WATER_TYPE_MAP = {
    'stream discharge': 'Stream Discharge',
    'groundwater':      'Groundwater',
    'reservoir':        'Reservoir',
    'irrigation':       'Irrigation',
    'springs':          'Springs',
    'precipitation':    'Precipitation',
    'water quality':    'Water Quality',
    'not water related':'Not Water Related',
    'other':            'Other',
}

def clean_water_type(wt):
    if pd.isna(wt):
        return 'Other'
    wt_lower = str(wt).strip().lower()
    for key, val in WATER_TYPE_MAP.items():
        if key in wt_lower:
            return val
    return 'Other'

df['water_type_clean'] = df['water_type'].map(clean_water_type)
print(df['water_type_clean'].value_counts())

water_type_clean
Stream Discharge     76060
Groundwater          20590
Reservoir             4384
Springs               2782
Irrigation            2383
Other                 1228
Precipitation          908
Not Water Related      758
Water Quality          347
Name: count, dtype: int64


## Locations

Rebuilt location hierarchy (agreed 2026-10-28; `code/01_digitization/04_metadata_extraction/location_validation/`).
LLM-inferred coordinates and LLM "actual" coordinates that are not printed on the page are **not used**.
`lat_combined`/`lon_combined` now hold the new locations; `coord_method` says how each was found,
`precision_class` is precise (HUC12-usable) / approximate (county/HUC8 scale) / unlocated, and
`evidence_scope` is page / document / continuation. The original LLM values are kept as `lat_llm`/`lon_llm`
(original pipeline preserved in `archive/2026-10-01_original_pipeline/`).

In [4]:
# original LLM coordinates, kept for reference only
df['lat_llm'] = df['actual_latitude'].combine_first(df['inferred_latitude'])
df['lon_llm'] = df['actual_longitude'].combine_first(df['inferred_longitude'])
df['coord_source_llm'] = np.where(df['actual_latitude'].notna(), 'actual',
                                  np.where(df['inferred_latitude'].notna(), 'inferred', None))

loc = pd.read_csv(root / 'data/analysis/location_validation/locations.csv', low_memory=False,
                  usecols=['meta_row', 'lat', 'lon', 'coord_method', 'precision_km',
                           'precision_class', 'evidence_scope'])
df = df.merge(loc.rename(columns={'lat': 'lat_combined', 'lon': 'lon_combined'}),
              on='meta_row', how='left', validate='one_to_one')
df.loc[df['coord_method'].eq('unlocated'), ['lat_combined', 'lon_combined']] = np.nan
df['coord_source'] = df['coord_method']   # kept for older code that reads coord_source

print(df['precision_class'].value_counts().to_string())
print(pd.crosstab(df['precision_class'], df['evidence_scope'].fillna('-')).to_string())

# page type (code/02_inventory/10_page_types.py): index/contents and narrative pages are
# kept but excluded from maps and statistics (is_data_page == False)
pt = pd.read_csv(root / 'data/analysis/page_types.csv', usecols=['id', 'page_number', 'page_type'])
df = df.merge(pt, on=['id', 'page_number'], how='left')
df['page_type'] = df['page_type'].fillna('no_page_text')
df['is_data_page'] = ~df['page_type'].isin(['index_contents', 'narrative'])
print(df['page_type'].value_counts().to_string())


precision_class
precise        54043
unlocated      34476
approximate    20128
basin            793
evidence_scope       -  continuation  document   page
precision_class                                      
approximate          0           159       578  19391
basin                0             0         0    793
precise              0             2      1729  52312
unlocated        34476             0         0      0
page_type
data              104954
index_contents      2995
no_page_text        1420
narrative             71


## Join measurement row counts

`csv_row_counts.csv` has one row per CSV table file. Aggregate to page level (sum across tables on the same page), then join to the metadata.

In [5]:
counts = pd.read_csv(counts_path)
print('Row counts columns:', counts.columns.tolist())

# Aggregate to page level
counts_page = (
    counts.groupby(['doc_id', 'page_number'])['number_rows']
    .sum()
    .reset_index()
    .rename(columns={'doc_id': 'id', 'number_rows': 'n_measurements'})
)

df = df.merge(counts_page, on=['id', 'page_number'], how='left')
matched = df['n_measurements'].notna().sum()
print(f'Matched row counts for {matched:,} of {len(df):,} rows')

Row counts columns: ['doc_id', 'page_number', 'table_number', 'number_rows']


Matched row counts for 106,734 of 109,440 rows


## Save

In [6]:
output_path.parent.mkdir(parents=True, exist_ok=True)
# Reviewer corrections (manual_review/earliest_records_review.csv): water type and dates.
# 'drop' verdicts only exclude pages from paper claims (07/08), so rows are kept here.
import sys; sys.path.insert(0, str(root / 'code/02_inventory'))
from paper_filters import apply_review
df = apply_review(df, drop=False)

df.to_parquet(output_path, index=False)
print(f'Saved {len(df):,} rows to {output_path}')
print('\nColumn summary:')
print(df[['id','page_number','year_start','year_end','decade_start',
          'water_type_clean','lat_combined','lon_combined',
          'coord_method','precision_class','n_measurements']].describe(include='all'))

  manual review applied: drop 1, retype 12, undate 40, construction_year 5, set_year 3, unlocate 24, keep 2 rows


Saved 109,440 rows to ..\..\data\analysis\processed_metadata.parquet

Column summary:
                  id    page_number   year_start     year_end  decade_start  \
count   1.094390e+05  109432.000000      93481.0      93481.0       93481.0   
unique           NaN            NaN         <NA>         <NA>          <NA>   
top              NaN            NaN         <NA>         <NA>          <NA>   
freq             NaN            NaN         <NA>         <NA>          <NA>   
mean    9.554302e+06     218.405905  1944.796044  1951.186915   1940.441266   
std     2.402125e+07     180.473698    26.378813    22.497686     26.381572   
min     1.500000e+01       1.000000       1800.0       1805.0        1800.0   
25%     1.595000e+03      63.000000       1923.0       1937.0        1920.0   
50%     3.090000e+03     172.000000       1952.0       1959.0        1950.0   
75%     5.636300e+04     340.000000       1966.0       1969.0        1960.0   
max     7.023307e+07     959.000000       198